# PERSUADE v8: Enhanced Span Ablation Analysis

**Primary influence metrics:**
1. **ΔNLL** (ablation – baseline): "how much predictive support came from that span"
2. **Δlogprob of actual token**: how much removing span S reduced probability of words actually used
3. **Rank displacement**: baseline rank of true token vs ablated rank

**Secondary diagnostics** (demoted): top-k Jaccard, flip rate

**Influence profiles:**
- Raw ΔNLL per span × score_region
- Normalized within-essay (proportion of total influence from each span)

**Dose response:** Run with span_size=128 and span_size=256

**Group analysis:** Compare score bins on normalized profiles and raw ΔNLL (controlling for fluency)

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes pandas numpy matplotlib seaborn tqdm statsmodels

In [ ]:
import json
import math
import os
import time
import random
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
import statsmodels.formula.api as smf

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Paths
DRIVE_BASE = "/content/drive/MyDrive/LRTIA/Data/persuade_clean"
OUTPUT_BASE = "/content/drive/MyDrive/LRTIA/Results/Persuade"
COHORT_PATH = f"{DRIVE_BASE}/cohorts/persuade_score_long_cohort.jsonl"

EXPERIMENT = 'score_long_span_ablation_v2'

# Model
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = False  # False for A100, True for T4

# Context/scoring parameters
CONTEXT_LENGTH = 512  # EXTENDED regime burn-in
TOTAL_SCORE_TOKENS = 256  # tokens 512-768

# Stratified scoring regions
SCORE_REGIONS = {
    'score_early': (512, 576),   # first 64 tokens after context
    'score_mid': (576, 640),     # next 64 tokens
    'score_late': (640, 768),    # final 128 tokens
}

# Span locations (as fraction of 512-token context)
SPAN_CONFIGS = {
    'early': (0.00, 0.10),              # tokens 0-51
    'early_mid': (0.10, 0.20),          # tokens 51-102
    'middle': (0.40, 0.50),             # tokens 205-256
    'late': (0.70, 0.80),               # tokens 358-409
    'pre_target': (0.80, 0.90),         # tokens 409-460
    'immediate_pre_target': (0.90, 1.00),  # tokens 460-512
}

# Ablation span sizes to test (DOSE RESPONSE)
SPAN_SIZES = [128, 256]  # Primary: 128, Dose: 256

# Random ablation settings
N_RANDOM_SPANS = 10  # Number of random spans per essay

# Secondary metrics
TOP_K = 10

RANDOM_SEED = 42

print(f"Experiment: {EXPERIMENT}")
print(f"Cohort: {COHORT_PATH}")
print(f"\nContext: {CONTEXT_LENGTH} tokens")
print(f"Scoring regions: {SCORE_REGIONS}")
print(f"\nSpan locations: {list(SPAN_CONFIGS.keys())} + random")
print(f"Span sizes (dose response): {SPAN_SIZES}")
print(f"Random spans per essay: {N_RANDOM_SPANS}")

## 1. Load Model and Data

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load model
print(f"Loading model: {MODEL_NAME}")

if USE_4BIT:
    print("  Using 4-bit quantization (T4 mode)")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    print("  Using float16 (A100 mode)")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )

model.eval()
print("Model loaded")

In [ ]:
# Load cohort
def load_cohort(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

cohort = load_cohort(COHORT_PATH)
print(f"Loaded {len(cohort)} essays")

# Verify cohort
df_cohort = pd.DataFrame(cohort)
print(f"\nScore bin distribution:")
print(df_cohort['score_bin'].value_counts())

## 2. Core Functions

In [ ]:
@torch.no_grad()
def get_logits(token_ids):
    """
    Get logits for all positions in token_ids.
    """
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    return outputs.logits[0]  # shape: (seq_len, vocab_size)


def compute_metrics_for_region(logits, token_ids, target_start, target_end):
    """
    Compute primary influence metrics for a scoring region.
    
    Primary metrics:
    - NLL per token (= -logprob of actual token)
    - Rank of actual token in distribution
    
    Secondary metrics:
    - Top-1 prediction
    - Top-k set
    
    Returns:
        dict with nlls, ranks, top1_preds, topk_sets, true_tokens
    """
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return {'nlls': [], 'ranks': [], 'top1_preds': [], 'topk_sets': [], 'true_tokens': []}
    
    nlls = []
    ranks = []
    top1_preds = []
    topk_sets = []
    true_tokens = []
    
    for i in range(target_start, target_end - 1):
        if i >= logits.shape[0]:
            break
        
        true_token = token_ids[i + 1]
        true_tokens.append(true_token)
        
        # NLL = -logprob of actual token
        log_probs = torch.log_softmax(logits[i], dim=-1)
        nll = -log_probs[true_token].item()
        nlls.append(nll)
        
        # Rank of actual token (0 = most likely)
        # Sort descending, find position of true token
        sorted_indices = logits[i].argsort(descending=True)
        rank = (sorted_indices == true_token).nonzero(as_tuple=True)[0].item()
        ranks.append(rank)
        
        # Secondary: Top-1 prediction
        top1 = logits[i].argmax().item()
        top1_preds.append(top1)
        
        # Secondary: Top-k set
        topk = set(logits[i].topk(TOP_K).indices.tolist())
        topk_sets.append(topk)
    
    return {
        'nlls': nlls,
        'ranks': ranks,
        'top1_preds': top1_preds,
        'topk_sets': topk_sets,
        'true_tokens': true_tokens,
    }


def compute_comparison_metrics(baseline, ablated):
    """
    Compare baseline vs ablated predictions.
    
    Primary metrics:
    - delta_nll: mean(ablated NLL) - mean(baseline NLL)
    - delta_logprob: same as delta_nll (NLL = -logprob)
    - rank_displacement: mean(ablated rank) - mean(baseline rank)
    
    Secondary metrics:
    - flip_rate: fraction where top-1 changes
    - topk_change: mean Jaccard distance of top-k sets
    
    Returns:
        dict with all metrics
    """
    if not baseline['nlls'] or not ablated['nlls']:
        return None
    
    n = min(len(baseline['nlls']), len(ablated['nlls']))
    
    # PRIMARY: Delta NLL (= -delta_logprob)
    baseline_nll_mean = np.mean(baseline['nlls'][:n])
    ablated_nll_mean = np.mean(ablated['nlls'][:n])
    delta_nll = ablated_nll_mean - baseline_nll_mean
    
    # PRIMARY: Rank displacement
    baseline_rank_mean = np.mean(baseline['ranks'][:n])
    ablated_rank_mean = np.mean(ablated['ranks'][:n])
    rank_displacement = ablated_rank_mean - baseline_rank_mean
    
    # Also compute median rank shift
    rank_shifts = [ablated['ranks'][i] - baseline['ranks'][i] for i in range(n)]
    rank_displacement_median = np.median(rank_shifts)
    
    # SECONDARY: Flip rate
    n_flips = sum(1 for i in range(n) if baseline['top1_preds'][i] != ablated['top1_preds'][i])
    flip_rate = n_flips / n
    
    # SECONDARY: Top-k Jaccard change
    topk_jaccard_changes = []
    for i in range(n):
        b_set = baseline['topk_sets'][i]
        a_set = ablated['topk_sets'][i]
        intersection = len(b_set & a_set)
        union = len(b_set | a_set)
        jaccard_sim = intersection / union if union > 0 else 1.0
        topk_jaccard_changes.append(1.0 - jaccard_sim)
    topk_change = np.mean(topk_jaccard_changes)
    
    return {
        # Primary
        'delta_nll': delta_nll,
        'rank_displacement_mean': rank_displacement,
        'rank_displacement_median': rank_displacement_median,
        # Sanity check values
        'baseline_nll': baseline_nll_mean,
        'ablated_nll': ablated_nll_mean,
        'baseline_rank_mean': baseline_rank_mean,
        'ablated_rank_mean': ablated_rank_mean,
        # Secondary
        'flip_rate': flip_rate,
        'topk_change': topk_change,
        'n_tokens': n,
    }


def ablate_span_deletion(token_ids, span_start, span_end):
    """
    Delete span and concatenate remaining tokens.
    """
    return token_ids[:span_start] + token_ids[span_end:]


print("Core functions defined")

In [ ]:
def compute_span_boundaries_fixed_size(context_length, frac_start, span_size):
    """
    Compute span boundaries with fixed size starting at fractional position.
    """
    span_start = int(context_length * frac_start)
    span_end = span_start + span_size
    
    # Clamp to context bounds
    if span_end > context_length:
        span_end = context_length
        span_start = max(0, span_end - span_size)
    
    return span_start, span_end


def run_span_ablation_for_essay(token_ids, essay_id, span_size, rng):
    """
    Run span ablation analysis for a single essay with stratified scoring.
    
    Returns:
        list of result dicts (one per span × score_region combination)
    """
    n_tokens = len(token_ids)
    
    # Need enough tokens for context + full scoring region
    min_required = CONTEXT_LENGTH + TOTAL_SCORE_TOKENS
    if n_tokens < min_required:
        return []
    
    # Full sequence for scoring: context + scoring region
    full_seq = token_ids[:min_required]
    
    # ============================================================
    # Baseline: full context
    # ============================================================
    baseline_logits = get_logits(full_seq)
    
    # Compute baseline metrics for each scoring region
    baseline_by_region = {}
    for region_name, (reg_start, reg_end) in SCORE_REGIONS.items():
        baseline_by_region[region_name] = compute_metrics_for_region(
            baseline_logits, full_seq, reg_start, reg_end
        )
    
    results = []
    
    # ============================================================
    # Fixed-position spans
    # ============================================================
    for span_label, (frac_start, frac_end) in SPAN_CONFIGS.items():
        span_start, span_end = compute_span_boundaries_fixed_size(
            CONTEXT_LENGTH, frac_start, span_size
        )
        actual_span_size = span_end - span_start
        
        # Ablate context
        ablated_context = ablate_span_deletion(full_seq[:CONTEXT_LENGTH], span_start, span_end)
        
        # Reconstruct: ablated context + original scoring tokens
        ablated_full = ablated_context + full_seq[CONTEXT_LENGTH:]
        
        # Get ablated logits
        ablated_logits = get_logits(ablated_full)
        
        # Compute metrics for each scoring region
        shift = CONTEXT_LENGTH - len(ablated_context)
        
        for region_name, (reg_start, reg_end) in SCORE_REGIONS.items():
            adj_start = reg_start - shift
            adj_end = reg_end - shift
            
            ablated_metrics = compute_metrics_for_region(
                ablated_logits, ablated_full, adj_start, adj_end
            )
            
            comparison = compute_comparison_metrics(
                baseline_by_region[region_name],
                ablated_metrics
            )
            
            if comparison:
                results.append({
                    'essay_id': essay_id,
                    'span_label': span_label,
                    'span_size': actual_span_size,
                    'span_start_token': span_start,
                    'span_end_token': span_end,
                    'score_region': region_name,
                    **comparison,
                })
    
    # ============================================================
    # Random spans (N_RANDOM_SPANS per essay, averaged later)
    # ============================================================
    random_results_by_region = defaultdict(list)
    
    for rand_idx in range(N_RANDOM_SPANS):
        max_start = CONTEXT_LENGTH - span_size
        if max_start <= 0:
            continue
        span_start = rng.randint(0, max_start)
        span_end = span_start + span_size
        
        ablated_context = ablate_span_deletion(full_seq[:CONTEXT_LENGTH], span_start, span_end)
        ablated_full = ablated_context + full_seq[CONTEXT_LENGTH:]
        ablated_logits = get_logits(ablated_full)
        
        shift = CONTEXT_LENGTH - len(ablated_context)
        
        for region_name, (reg_start, reg_end) in SCORE_REGIONS.items():
            adj_start = reg_start - shift
            adj_end = reg_end - shift
            
            ablated_metrics = compute_metrics_for_region(
                ablated_logits, ablated_full, adj_start, adj_end
            )
            
            comparison = compute_comparison_metrics(
                baseline_by_region[region_name],
                ablated_metrics
            )
            
            if comparison:
                random_results_by_region[region_name].append(comparison)
    
    # Average random results within essay
    for region_name, comparisons in random_results_by_region.items():
        if comparisons:
            results.append({
                'essay_id': essay_id,
                'span_label': 'random',
                'span_size': span_size,
                'span_start_token': np.nan,
                'span_end_token': np.nan,
                'score_region': region_name,
                # Average all metrics
                'delta_nll': np.mean([c['delta_nll'] for c in comparisons]),
                'rank_displacement_mean': np.mean([c['rank_displacement_mean'] for c in comparisons]),
                'rank_displacement_median': np.median([c['rank_displacement_median'] for c in comparisons]),
                'baseline_nll': np.mean([c['baseline_nll'] for c in comparisons]),
                'ablated_nll': np.mean([c['ablated_nll'] for c in comparisons]),
                'baseline_rank_mean': np.mean([c['baseline_rank_mean'] for c in comparisons]),
                'ablated_rank_mean': np.mean([c['ablated_rank_mean'] for c in comparisons]),
                'flip_rate': np.mean([c['flip_rate'] for c in comparisons]),
                'topk_change': np.mean([c['topk_change'] for c in comparisons]),
                'n_tokens': comparisons[0]['n_tokens'],
                'n_random_samples': len(comparisons),
            })
    
    return results


print("Span ablation function defined")

## 3. Run Span Ablation (Dose Response: 128 and 256)

In [ ]:
# Pre-tokenize all essays
essay_tokens = {}
for essay in cohort:
    token_ids = tokenizer.encode(essay['text'], add_special_tokens=False)
    essay_tokens[essay['essay_id']] = token_ids

print(f"Tokenized {len(essay_tokens)} essays")

# Token length stats
lengths = [len(t) for t in essay_tokens.values()]
print(f"Token lengths: min={min(lengths)}, median={np.median(lengths):.0f}, max={max(lengths)}")

In [ ]:
# Process all essays with DOSE RESPONSE
all_results = []
start_time = time.time()

for span_size in SPAN_SIZES:
    print(f"\n{'='*60}")
    print(f"DOSE: span_size = {span_size} tokens")
    print(f"{'='*60}")
    
    rng = random.Random(RANDOM_SEED)
    
    n_spans = len(SPAN_CONFIGS) + 1  # +1 for random
    n_regions = len(SCORE_REGIONS)
    expected_rows = len(cohort) * n_spans * n_regions
    print(f"Expected: {len(cohort)} essays × {n_spans} spans × {n_regions} regions = {expected_rows} rows")
    
    for essay in tqdm(cohort, desc=f"Span size {span_size}"):
        essay_id = essay['essay_id']
        token_ids = essay_tokens[essay_id]
        
        results = run_span_ablation_for_essay(token_ids, essay_id, span_size, rng)
        
        for r in results:
            r['score'] = essay.get('score')
            r['score_bin'] = essay.get('score_bin')
            r['grade'] = essay.get('grade')
            r['token_count'] = len(token_ids)
        
        all_results.extend(results)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f}s ({elapsed/len(cohort)/len(SPAN_SIZES):.2f}s/essay/dose)")
print(f"Total result rows: {len(all_results)}")

In [ ]:
# Create DataFrame
df = pd.DataFrame(all_results)

print(f"Results shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSpan size × Span label counts:")
print(pd.crosstab(df['span_size'], df['span_label']))

df.head(10)

## 4. Sanity Checks

In [ ]:
# SANITY CHECK: Baseline and ablated NLL ranges
print("="*80)
print("SANITY CHECK: NLL Ranges (ensure deltas aren't saturating or tiny)")
print("="*80)

for span_size in SPAN_SIZES:
    subset = df[df['span_size'] == span_size]
    print(f"\n--- Span size {span_size} ---")
    print(f"Baseline NLL: mean={subset['baseline_nll'].mean():.3f}, "
          f"std={subset['baseline_nll'].std():.3f}, "
          f"range=[{subset['baseline_nll'].min():.3f}, {subset['baseline_nll'].max():.3f}]")
    print(f"Ablated NLL:  mean={subset['ablated_nll'].mean():.3f}, "
          f"std={subset['ablated_nll'].std():.3f}, "
          f"range=[{subset['ablated_nll'].min():.3f}, {subset['ablated_nll'].max():.3f}]")
    print(f"Delta NLL:    mean={subset['delta_nll'].mean():.4f}, "
          f"std={subset['delta_nll'].std():.4f}, "
          f"range=[{subset['delta_nll'].min():.4f}, {subset['delta_nll'].max():.4f}]")
    print(f"Rank displacement (mean): mean={subset['rank_displacement_mean'].mean():.1f}, "
          f"range=[{subset['rank_displacement_mean'].min():.1f}, {subset['rank_displacement_mean'].max():.1f}]")

In [ ]:
# SANITY CHECK: Delta distributions by span
print("\n" + "="*80)
print("SANITY CHECK: Mean ΔNLL by Span Label (should be positive, larger for boundary-adjacent)")
print("="*80)

for span_size in SPAN_SIZES:
    print(f"\n--- Span size {span_size} ---")
    subset = df[df['span_size'] == span_size]
    summary = subset.groupby('span_label')['delta_nll'].agg(['mean', 'std', 'count'])
    print(summary.round(4))

## 5. Build Influence Profiles

In [ ]:
# Define ordering
GROUP_ORDER = ['low', 'mid', 'high']
SPAN_ORDER = ['early', 'early_mid', 'middle', 'late', 'pre_target', 'immediate_pre_target', 'random']
REGION_ORDER = ['score_early', 'score_mid', 'score_late']

# Set categorical ordering
df['score_bin'] = pd.Categorical(df['score_bin'], categories=GROUP_ORDER, ordered=True)
df['span_label'] = pd.Categorical(df['span_label'], categories=SPAN_ORDER, ordered=True)
df['score_region'] = pd.Categorical(df['score_region'], categories=REGION_ORDER, ordered=True)

In [ ]:
# Compute NORMALIZED influence profiles per essay
# For each essay: divide each span's ΔNLL by total ΔNLL across all spans
# This shows WHERE influence comes from, not how hard the essay is

def compute_normalized_influence(group):
    """
    Normalize delta_nll within an essay (for a given span_size and score_region).
    """
    # Only use fixed spans for normalization (exclude random)
    fixed_spans = group[group['span_label'] != 'random']
    
    # Total positive influence (sum of positive deltas)
    # Use absolute values to handle any negative deltas
    total_influence = fixed_spans['delta_nll'].abs().sum()
    
    if total_influence > 0:
        group = group.copy()
        group['delta_nll_normalized'] = group['delta_nll'] / total_influence
    else:
        group = group.copy()
        group['delta_nll_normalized'] = 0.0
    
    return group

# Apply normalization per essay × span_size × score_region
df = df.groupby(['essay_id', 'span_size', 'score_region'], group_keys=False).apply(
    compute_normalized_influence
)

print("Normalized influence computed")
print(f"\nSample normalized values (span_size=128, score_early):")
sample = df[(df['span_size'] == 128) & (df['score_region'] == 'score_early')]
print(sample.groupby('span_label')['delta_nll_normalized'].mean().round(4))

## 6. Group Summaries

In [ ]:
# Summary by score_bin × span_label × score_region × span_size
print("="*80)
print("PRIMARY METRIC: ΔNLL BY SCORE_BIN × SPAN_LABEL × SCORE_REGION")
print("="*80)

summary_rows = []

for span_size in SPAN_SIZES:
    for score_bin in GROUP_ORDER:
        for span_label in SPAN_ORDER:
            for region in REGION_ORDER:
                subset = df[(df['span_size'] == span_size) &
                           (df['score_bin'] == score_bin) & 
                           (df['span_label'] == span_label) & 
                           (df['score_region'] == region)]
                if len(subset) == 0:
                    continue
                
                summary_rows.append({
                    'span_size': span_size,
                    'score_bin': score_bin,
                    'span_label': span_label,
                    'score_region': region,
                    'n': len(subset),
                    # Primary
                    'delta_nll_mean': subset['delta_nll'].mean(),
                    'delta_nll_sem': subset['delta_nll'].sem(),
                    'delta_nll_normalized_mean': subset['delta_nll_normalized'].mean(),
                    'rank_displacement_mean': subset['rank_displacement_mean'].mean(),
                    'rank_displacement_median': subset['rank_displacement_median'].median(),
                    # Sanity
                    'baseline_nll_mean': subset['baseline_nll'].mean(),
                    'ablated_nll_mean': subset['ablated_nll'].mean(),
                    # Secondary
                    'flip_rate_mean': subset['flip_rate'].mean(),
                    'topk_change_mean': subset['topk_change'].mean(),
                })

df_summary = pd.DataFrame(summary_rows)
print(f"Summary table: {len(df_summary)} rows")

In [ ]:
# KEY TABLE: Pivoted view for span_size=128
print("\n" + "="*80)
print("PIVOT: Mean ΔNLL (span_size=128)")
print("="*80)

df_128 = df[df['span_size'] == 128]

for score_bin in GROUP_ORDER:
    print(f"\n--- Score bin: {score_bin} ---")
    subset = df_128[df_128['score_bin'] == score_bin]
    pivot = subset.pivot_table(
        values='delta_nll',
        index='span_label',
        columns='score_region',
        aggfunc='mean'
    )
    if REGION_ORDER[0] in pivot.columns:
        pivot = pivot[REGION_ORDER]
    print(pivot.round(4))

In [ ]:
# NORMALIZED influence profiles
print("\n" + "="*80)
print("PIVOT: Normalized Influence (within-essay proportions, span_size=128)")
print("="*80)

for score_bin in GROUP_ORDER:
    print(f"\n--- Score bin: {score_bin} ---")
    subset = df_128[(df_128['score_bin'] == score_bin) & (df_128['span_label'] != 'random')]
    pivot = subset.pivot_table(
        values='delta_nll_normalized',
        index='span_label',
        columns='score_region',
        aggfunc='mean'
    )
    if REGION_ORDER[0] in pivot.columns:
        pivot = pivot[REGION_ORDER]
    print(pivot.round(4))

In [ ]:
# DOSE RESPONSE: Compare span_size 128 vs 256
print("\n" + "="*80)
print("DOSE RESPONSE: Span Size 128 vs 256")
print("="*80)
print("Does the influence profile SHAPE change with larger ablation?")

# Compare normalized profiles
for region in REGION_ORDER:
    print(f"\n--- {region} ---")
    print(f"{'Span':<25} {'Size 128':>12} {'Size 256':>12} {'Ratio':>10}")
    print("-"*65)
    
    for span_label in SPAN_ORDER[:-1]:  # Exclude random
        val_128 = df_summary[(df_summary['span_size'] == 128) & 
                            (df_summary['span_label'] == span_label) & 
                            (df_summary['score_region'] == region)]['delta_nll_mean'].mean()
        val_256 = df_summary[(df_summary['span_size'] == 256) & 
                            (df_summary['span_label'] == span_label) & 
                            (df_summary['score_region'] == region)]['delta_nll_mean'].mean()
        ratio = val_256 / val_128 if val_128 != 0 else np.nan
        print(f"{span_label:<25} {val_128:>12.4f} {val_256:>12.4f} {ratio:>10.2f}")

## 7. Statistical Tests

In [ ]:
# Standardize controls
df['token_count_z'] = (df['token_count'] - df['token_count'].mean()) / df['token_count'].std()
df['baseline_nll_z'] = (df['baseline_nll'] - df['baseline_nll'].mean()) / df['baseline_nll'].std()

# Use span_size=128 for primary analysis
df_128 = df[df['span_size'] == 128].copy()

print("="*80)
print("REGRESSION: delta_nll ~ span + region + score_bin + controls + interactions")
print("="*80)

# Full model with fluency control
formula = ('delta_nll ~ C(span_label) + C(score_region) + C(score_bin) + '
           'C(span_label):C(score_region) + C(span_label):C(score_bin) + '
           'token_count_z + baseline_nll_z')

model_full = smf.ols(formula, data=df_128).fit()

print(f"\nFormula: {formula}")
print(f"R²: {model_full.rsquared:.4f}, Adj R²: {model_full.rsquared_adj:.4f}, n={int(model_full.nobs)}")

In [ ]:
# Key coefficients: span effects
print("\n" + "="*80)
print("SPAN EFFECTS (vs early)")
print("="*80)

span_params = [p for p in model_full.params.index if 'span_label' in p and ':' not in p]
for param in sorted(span_params):
    coef = model_full.params[param]
    pval = model_full.pvalues[param]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

In [ ]:
# Key test: span × region interaction
print("\n" + "="*80)
print("SPAN × REGION INTERACTION (does span influence vary by distance to target?)")
print("="*80)

interaction_params = [p for p in model_full.params.index if 'span_label' in p and 'score_region' in p]
print(f"\n{len(interaction_params)} interaction terms:")
for param in sorted(interaction_params)[:10]:  # Show first 10
    coef = model_full.params[param]
    pval = model_full.pvalues[param]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

In [ ]:
# Test: Does score_bin affect normalized influence profiles?
print("\n" + "="*80)
print("GROUP ANALYSIS: Do score bins differ in WHERE influence comes from?")
print("="*80)

# Use normalized influence (excludes random)
df_norm = df_128[df_128['span_label'] != 'random'].copy()

formula_norm = ('delta_nll_normalized ~ C(span_label) + C(score_region) + C(score_bin) + '
                'C(span_label):C(score_bin)')

model_norm = smf.ols(formula_norm, data=df_norm).fit()

print(f"\nNormalized influence model:")
print(f"R²: {model_norm.rsquared:.4f}")

# Score bin × span interactions
score_span_params = [p for p in model_norm.params.index if 'score_bin' in p and 'span_label' in p]
print(f"\nScore bin × Span interactions (do groups use context differently?):")
sig_count = 0
for param in sorted(score_span_params):
    coef = model_norm.params[param]
    pval = model_norm.pvalues[param]
    if pval < 0.05:
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*"
        print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")
        sig_count += 1

if sig_count == 0:
    print("  No significant interactions (groups use context similarly)")

In [ ]:
# Raw ΔNLL comparison controlling for fluency
print("\n" + "="*80)
print("GROUP ANALYSIS: Raw ΔNLL (absolute influence) controlling for fluency")
print("="*80)

# Model with fluency control
formula_raw = 'delta_nll ~ C(score_bin) + baseline_nll_z + token_count_z'
model_raw = smf.ols(formula_raw, data=df_128).fit()

print(f"\nModel: {formula_raw}")
print(f"R²: {model_raw.rsquared:.4f}")
print(f"\nScore bin effects (controlling for fluency):")
for param in ['C(score_bin)[T.mid]', 'C(score_bin)[T.high]']:
    if param in model_raw.params:
        coef = model_raw.params[param]
        pval = model_raw.pvalues[param]
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

print(f"\nFluency control (baseline_nll_z):")
print(f"  β={model_raw.params['baseline_nll_z']:+.4f}, p={model_raw.pvalues['baseline_nll_z']:.4f}")

## 8. Visualizations

In [ ]:
# Color schemes
SCORE_COLORS = {'low': '#e74c3c', 'mid': '#f39c12', 'high': '#2ecc71'}
REGION_COLORS = {'score_early': '#3498db', 'score_mid': '#9b59b6', 'score_late': '#1abc9c'}

In [ ]:
# KEY PLOT 1: Heatmap of raw ΔNLL (span × region)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Raw ΔNLL
ax = axes[0]
pivot_raw = df_128.pivot_table(
    values='delta_nll',
    index='span_label',
    columns='score_region',
    aggfunc='mean'
)[REGION_ORDER]

sns.heatmap(pivot_raw, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'ΔNLL'})
ax.set_title('Raw ΔNLL: Span Location × Scoring Region\n(Higher = more influence)', 
             fontsize=11, fontweight='bold')
ax.set_xlabel('Scoring Region')
ax.set_ylabel('Ablated Span')

# Normalized influence
ax = axes[1]
pivot_norm = df_128[df_128['span_label'] != 'random'].pivot_table(
    values='delta_nll_normalized',
    index='span_label',
    columns='score_region',
    aggfunc='mean'
)[REGION_ORDER]

sns.heatmap(pivot_norm, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Normalized Influence'})
ax.set_title('Normalized Influence: Where Does Context Come From?\n(Within-essay proportions)', 
             fontsize=11, fontweight='bold')
ax.set_xlabel('Scoring Region')
ax.set_ylabel('Ablated Span')

plt.tight_layout()
plt.savefig(Path(OUTPUT_BASE) / EXPERIMENT / 'influence_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# KEY PLOT 2: Influence profiles by score bin
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

x_positions = np.arange(len(SPAN_ORDER) - 1)  # Exclude random
width = 0.25

for ax_idx, region in enumerate(REGION_ORDER):
    ax = axes[ax_idx]
    region_data = df_128[(df_128['score_region'] == region) & (df_128['span_label'] != 'random')]
    
    for i, score_bin in enumerate(GROUP_ORDER):
        means = []
        cis = []
        for span in SPAN_ORDER[:-1]:
            subset = region_data[(region_data['score_bin'] == score_bin) & 
                                (region_data['span_label'] == span)]
            if len(subset) > 0:
                means.append(subset['delta_nll'].mean())
                cis.append(1.96 * subset['delta_nll'].sem())
            else:
                means.append(np.nan)
                cis.append(np.nan)
        
        offset = (i - 1) * width
        ax.bar(x_positions + offset, means, width, yerr=cis,
               label=score_bin, color=SCORE_COLORS[score_bin], capsize=2, alpha=0.8)
    
    ax.set_title(f'{region}\n(tokens {SCORE_REGIONS[region][0]}-{SCORE_REGIONS[region][1]})', 
                 fontsize=11, fontweight='bold')
    ax.set_xticks(x_positions)
    ax.set_xticklabels(['early', 'early\nmid', 'middle', 'late', 'pre\ntarget', 'immed\npre'],
                       fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
    
    if ax_idx == 0:
        ax.set_ylabel('ΔNLL (ablated - baseline)', fontsize=11)
    if ax_idx == 1:
        ax.legend(title='Score Bin', loc='upper left', fontsize=9)

fig.suptitle('Span Influence by Score Bin and Scoring Region (span_size=128)\n'
             '(Higher ΔNLL = more predictive support from that span)', 
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(Path(OUTPUT_BASE) / EXPERIMENT / 'influence_by_group.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# KEY PLOT 3: Decay of boundary-adjacent span influence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: immediate_pre_target effect by region
ax = axes[0]
imm_pre = df_128[df_128['span_label'] == 'immediate_pre_target']

for score_bin in GROUP_ORDER:
    means = []
    cis = []
    for region in REGION_ORDER:
        subset = imm_pre[(imm_pre['score_bin'] == score_bin) & (imm_pre['score_region'] == region)]
        if len(subset) > 0:
            means.append(subset['delta_nll'].mean())
            cis.append(1.96 * subset['delta_nll'].sem())
        else:
            means.append(np.nan)
            cis.append(np.nan)
    
    ax.errorbar(range(len(REGION_ORDER)), means, yerr=cis, marker='o', capsize=4,
                label=score_bin, color=SCORE_COLORS[score_bin], linewidth=2, markersize=8)

ax.set_xticks(range(len(REGION_ORDER)))
ax.set_xticklabels(['Early\n(512-576)', 'Mid\n(576-640)', 'Late\n(640-768)'])
ax.set_xlabel('Scoring Region (distance from ablated span)', fontsize=11)
ax.set_ylabel('ΔNLL', fontsize=11)
ax.set_title('Immediate Pre-Target Ablation:\nEffect Decay with Distance', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

# Plot 2: Rank displacement by span (score_early only)
ax = axes[1]
score_early = df_128[(df_128['score_region'] == 'score_early') & (df_128['span_label'] != 'random')]

for score_bin in GROUP_ORDER:
    means = []
    for span in SPAN_ORDER[:-1]:
        subset = score_early[(score_early['score_bin'] == score_bin) & 
                            (score_early['span_label'] == span)]
        if len(subset) > 0:
            means.append(subset['rank_displacement_mean'].mean())
        else:
            means.append(np.nan)
    
    ax.plot(range(6), means, marker='s', label=score_bin, 
            color=SCORE_COLORS[score_bin], linewidth=2, markersize=8)

ax.set_xticks(range(6))
ax.set_xticklabels(['early', 'early_mid', 'middle', 'late', 'pre_target', 'immed_pre'], rotation=45)
ax.set_xlabel('Ablated Span Location (left=distant, right=adjacent)', fontsize=11)
ax.set_ylabel('Rank Displacement (mean)', fontsize=11)
ax.set_title('score_early Region:\nRank Displacement by Span Distance', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(Path(OUTPUT_BASE) / EXPERIMENT / 'decay_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# KEY PLOT 4: Dose response comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax_idx, region in enumerate(REGION_ORDER):
    ax = axes[ax_idx]
    
    for span_size, marker, ls in [(128, 'o', '-'), (256, 's', '--')]:
        subset = df[(df['span_size'] == span_size) & 
                   (df['score_region'] == region) & 
                   (df['span_label'] != 'random')]
        
        means = []
        for span in SPAN_ORDER[:-1]:
            span_subset = subset[subset['span_label'] == span]
            if len(span_subset) > 0:
                means.append(span_subset['delta_nll'].mean())
            else:
                means.append(np.nan)
        
        ax.plot(range(6), means, marker=marker, linestyle=ls, 
                label=f'span={span_size}', linewidth=2, markersize=7)
    
    ax.set_xticks(range(6))
    ax.set_xticklabels(['early', 'early\nmid', 'middle', 'late', 'pre\ntgt', 'imm\npre'], fontsize=8)
    ax.set_title(f'{region}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    if ax_idx == 0:
        ax.set_ylabel('ΔNLL', fontsize=11)
        ax.legend(fontsize=9)

fig.suptitle('Dose Response: Span Size 128 vs 256\n(Does profile shape change with larger ablation?)', 
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(Path(OUTPUT_BASE) / EXPERIMENT / 'dose_response.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Results

In [ ]:
# Create output directory
output_dir = Path(OUTPUT_BASE) / EXPERIMENT
output_dir.mkdir(parents=True, exist_ok=True)

# Save full results
df.to_csv(output_dir / 'span_ablation_v2_results.csv', index=False)

# Save summary
df_summary.to_csv(output_dir / 'span_ablation_v2_summary.csv', index=False)

# Save pivot tables
pivot_raw.to_csv(output_dir / 'pivot_delta_nll_raw.csv')
pivot_norm.to_csv(output_dir / 'pivot_delta_nll_normalized.csv')

# Save regression results
with open(output_dir / 'regression_v2.txt', 'w') as f:
    f.write("ENHANCED SPAN ABLATION REGRESSION ANALYSIS (v2)\n")
    f.write("="*80 + "\n\n")
    f.write(f"Configuration:\n")
    f.write(f"  Span sizes (dose): {SPAN_SIZES}\n")
    f.write(f"  Span locations: {list(SPAN_CONFIGS.keys())} + random\n")
    f.write(f"  Scoring regions: {SCORE_REGIONS}\n")
    f.write(f"  Random samples per essay: {N_RANDOM_SPANS}\n")
    f.write("\n" + "="*80 + "\n\n")
    f.write("PRIMARY METRICS:\n")
    f.write("  1. ΔNLL (ablated - baseline): predictive support from span\n")
    f.write("  2. Rank displacement: how much true token rank worsens\n")
    f.write("  3. Normalized influence: within-essay proportion\n")
    f.write("\n" + "="*80 + "\n\n")
    f.write("Full Model (span_size=128):\n")
    f.write(f"Formula: {formula}\n")
    f.write(model_full.summary().as_text())
    f.write("\n\n" + "="*80 + "\n\n")
    f.write("Normalized Influence Model:\n")
    f.write(f"Formula: {formula_norm}\n")
    f.write(model_norm.summary().as_text())

print(f"\nSaved to {output_dir}/")
print(f"  - span_ablation_v2_results.csv ({len(df)} rows)")
print(f"  - span_ablation_v2_summary.csv")
print(f"  - pivot_delta_nll_raw.csv")
print(f"  - pivot_delta_nll_normalized.csv")
print(f"  - regression_v2.txt")
print(f"  - influence_heatmaps.png")
print(f"  - influence_by_group.png")
print(f"  - decay_analysis.png")
print(f"  - dose_response.png")

In [ ]:
# Final summary
print("\n" + "="*80)
print("SUMMARY OF FINDINGS")
print("="*80)

print("""
PRIMARY METRICS:
1. ΔNLL = ablated NLL - baseline NLL
   → Higher = more predictive support came from that span
   → Directly measures: "how much did removing span S reduce probability of actual words?"

2. Rank displacement = ablated rank - baseline rank
   → Higher = true token rank worsened more
   → Captures influence even when top-1 doesn't change

3. Normalized influence = span ΔNLL / total ΔNLL
   → Shows WHERE influence comes from (proportion)
   → Removes essay difficulty confound

KEY QUESTIONS:
1. Does boundary-adjacent context (immediate_pre_target) have highest impact on nearby tokens (score_early)?
   → Check: ΔNLL should be highest for immediate_pre_target on score_early
   → Check: Effect should decay (lower on score_mid, score_late)

2. Do score groups differ in WHERE influence comes from?
   → Check: Normalized influence profiles across span labels
   → If high-quality essays use distant context more: higher early/middle span influence

3. Is the pattern robust to dose (span size)?
   → Compare 128 vs 256 token ablations
   → Shape should be similar, magnitude larger for 256

SECONDARY DIAGNOSTICS (demoted):
- Flip rate (top-1 changes)
- Top-k Jaccard change
""")